In [2]:
using DelimitedFiles
using Statistics
using Ripserer
using ProgressMeter
using NPZ
using MAT

# Schaefer300

In [1]:
data_dir = "/Data/HCP/100unrelated/schaefer300/no_hp/"
input_dir = "ts/LR/"
filenames = readdir("$data_dir$input_dir")

woutput_dir = "/Data/HCP/100unrelated/schaefer300/no_hp/scaffolds/whole"

n = 100
p = ProgressMeter.Progress(n)

Threads.@threads for filename in filenames
    sub_name = filename[1:6]
    session = filename[12]-1
    if isfile("$woutput_dir/freq/$(sub_name)_s$(session)") 
    else
        input_path = "$data_dir$input_dir$(filename)"
        #read CSV
        ts = readdlm(input_path,',', skipstart=1)
        #compute corr matris
        pms = cor(ts)
        num_regions = size(pms)[2]
        #copmute distance correlation
        dist_corrs = 1 .- pms
        #symmetrize matrix
        dist_corrs = (dist_corrs .+ transpose(dist_corrs))/2
        #dist_corrs = replace(dist_corrs, NaN => 1.5)  in case there is some missing series
        d = ripserer((dist_corrs .+ transpose(dist_corrs))/2; reps=1, alg=:homology, verbose=false)
        p_scaffold_b = zeros(num_regions,num_regions)
        f_scaffold_b = zeros(num_regions,num_regions)
        for l in 1:length(d[2])
            g = d[2][l]
            pers = persistence(g)
            for rep in representative(g) 
                i , j = vertices(rep)
                p_scaffold_b[i,j] += pers
                f_scaffold_b[i,j] += 1
                p_scaffold_b[j,i] += pers
                f_scaffold_b[j,i] += 1
            end
        end
        npzwrite("$woutput_dir/freq/$(sub_name)_s$(session)", f_scaffold_b)
        npzwrite("$woutput_dir/pers/$(sub_name)_s$(session)", p_scaffold_b)
    end
    ProgressMeter.next!(p)
end

LoadError: UndefVarError: `ProgressMeter` not defined

# Sliding windows

In [ ]:
input_dir = "/Data/HCP/100unrelated/schaefer300/no_hp/fc/"
widths = [1100,1000,900,800,700,600,500,400,300,200,100,50,25,10];
woutput_dir = "/Data/HCP/100unrelated/schaefer300/no_hp/scaffolds/"

@showprogress for width in widths
    filenames = readdir("$input_dir$width/");
    n = 100
    p = ProgressMeter.Progress(n)

    Threads.@threads for filename in filenames
        sub_name = filename[1:6]
        if isfile("$woutput_dir/freq/$(sub_name)")
        else
            fcs = npzread("$input_dir$width/$filename")
            L = size(fcs)[1]
            num_regions = size(fcs)[2]
            dist_corrs = [ 1 .- fcs[i,:,:] for i in 1:L];
            diagrams = [ripserer((i .+ transpose(i))/2; reps=1, alg=:homology, verbose=false) for i in dist_corrs];
            p_tensor_scaffold_b = zeros(L,num_regions,num_regions)
            f_tensor_scaffold_b = zeros(L,num_regions,num_regions)
            for timeframe in 1:length(diagrams)
                d = diagrams[timeframe]
                for l in 1:length(d[2])
                    g = d[2][l]
                    pers = persistence(g)
                    for rep in representative(g) 
                        i , j = vertices(rep)
                        p_tensor_scaffold_b[timeframe,i,j] += pers
                        f_tensor_scaffold_b[timeframe,i,j] += 1
                        p_tensor_scaffold_b[timeframe,j,i] += pers
                        f_tensor_scaffold_b[timeframe,j,i] += 1
                    end
                end
            end
            npzwrite("$woutput_dir$width/freq/$(sub_name)", f_tensor_scaffold_b)
            npzwrite("$woutput_dir$width/pers/$(sub_name)", p_tensor_scaffold_b)
        end
        ProgressMeter.next!(p)
    end
end

Progress: 100%|█████████████████████████████████████████| Time: 0:05:26
Progress: 100%|█████████████████████████████████████████| Time: 0:07:40
Progress: 100%|█████████████████████████████████████████| Time: 0:10:30                                                                            
Progress: 100%|█████████████████████████████████████████| Time: 0:13:04
Progress: 100%|█████████████████████████████████████████| Time: 0:15:27
Progress: 100%|█████████████████████████████████████████| Time: 0:15:37
Progress: 100%|█████████████████████████████████████████| Time: 0:15:35
Progress: 100%|█████████████████████████████████████████| Time: 0:16:26
Progress: 100%|█████████████████████████████████████████| Time: 0:16:48
Progress:  64%|██████████████████████████▍              |  ETA: 1:04:48